# Mount St. Helens terrain model from a bounding box

Downloads a 10 m USGS 3DEP DEM for a lon/lat bounding box and turns it into a watertight binary STL with a solid base. The longer side of the model is 6 inches (152.4 mm), and the other side is scaled to keep the true aspect ratio. A hillshade preview image is also written.

Edit the **Configuration** cell below, then run all cells.

In [ ]:
import numpy as np, tifffile, struct, os, math, time, urllib.request, urllib.error
from PIL import Image
from pyproj import Transformer

## Configuration

- `BBOX`: `(lon_min, lat_min, lon_max, lat_max)` in WGS84 degrees
- `EXAG`: vertical exaggeration (1.0 = true scale)

The output name includes the exaggeration, e.g. `mount_st_helens_2.5x.stl`. The DEM is cached under a filename built from the bbox and resolution, so changing either triggers a new download.

In [ ]:
BBOX = (-122.30, 46.13, -122.10, 46.26)   # lon_min, lat_min, lon_max, lat_max (WGS84)
EXAG = 2.5                                 # vertical exaggeration

## Model parameters

In [ ]:
SIZE_MM = 152.4      # 6 inches along the longer side
GRID = 500           # samples along the longer side (~0.3 mm spacing)
BASE_MM = 3.0        # solid base below lowest point
RES_M = 10.0         # DEM resolution

assert BBOX[0] < BBOX[2] and BBOX[1] < BBOX[3], 'BBOX must be (lon_min, lat_min, lon_max, lat_max)'
NAME = f'mount_st_helens_{EXAG:g}x'
DEM_FILE = 'dem_{:.4f}_{:.4f}_{:.4f}_{:.4f}_{:g}m.tif'.format(*BBOX, RES_M)
print(NAME, DEM_FILE)

## Project the bounding box to UTM

The bbox corners are projected to the UTM zone of the bbox centre, and the enclosing rectangle is snapped to the DEM resolution. Because a lon/lat box is not a perfect rectangle in UTM, the downloaded area can extend slightly past the bbox on some edges.

In [ ]:
zone = int((BBOX[0] + BBOX[2]) / 2 // 6 + 31)
EPSG = (32600 if (BBOX[1] + BBOX[3]) / 2 >= 0 else 32700) + zone
t = Transformer.from_crs(4326, EPSG, always_xy=True)
ex, ny = t.transform([BBOX[0], BBOX[0], BBOX[2], BBOX[2]], [BBOX[1], BBOX[3], BBOX[1], BBOX[3]])
xmin, ymin = math.floor(min(ex) / RES_M) * RES_M, math.floor(min(ny) / RES_M) * RES_M
xmax, ymax = math.ceil(max(ex) / RES_M) * RES_M, math.ceil(max(ny) / RES_M) * RES_M
EXTENT_X, EXTENT_Y = xmax - xmin, ymax - ymin
W, H = int(EXTENT_X / RES_M), int(EXTENT_Y / RES_M)
print(f'EPSG:{EPSG}  extent {EXTENT_X/1000:.2f} x {EXTENT_Y/1000:.2f} km  ({W} x {H} px)')

## Download the DEM from USGS 3DEP

Skipped if the cached file already exists. The 3DEP ImageServer sometimes returns 504 errors, so the download retries up to 5 times.

In [ ]:
if not os.path.exists(DEM_FILE):
    url = ('https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer/exportImage'
           f'?bbox={xmin:.0f},{ymin:.0f},{xmax:.0f},{ymax:.0f}&bboxSR={EPSG}&imageSR={EPSG}&size={W},{H}'
           '&format=tiff&pixelType=F32&interpolation=RSP_BilinearInterpolation&f=image')
    for attempt in range(5):
        try:
            urllib.request.urlretrieve(url, DEM_FILE)
            break
        except urllib.error.HTTPError as e:
            if attempt == 4:
                raise
            print(f'download failed ({e.code}), retrying...')
            time.sleep(10 * (attempt + 1))

## Load and resample the DEM

Nodata values (< -1000 m) are filled with the minimum elevation, then the grid is resampled so the longer side has `GRID` samples.

In [ ]:
dem = tifffile.imread(DEM_FILE).astype(np.float64)
dem = np.where(dem < -1000, np.nan, dem)
dem = np.nan_to_num(dem, nan=np.nanmin(dem))
scale = SIZE_MM / max(EXTENT_X, EXTENT_Y)   # mm per meter horizontally
GX = round(GRID * EXTENT_X / max(EXTENT_X, EXTENT_Y))
GY = round(GRID * EXTENT_Y / max(EXTENT_X, EXTENT_Y))
img = Image.fromarray(dem.astype(np.float32)).resize((GX, GY), Image.BILINEAR)
z_m = np.array(img, dtype=np.float64)
print(f'elevation {dem.min():.0f}-{dem.max():.0f} m, grid {GX} x {GY}')

## Scale to millimetres

Elevations use the same horizontal scale, multiplied by `EXAG` and raised by `BASE_MM`.

In [ ]:
z = (z_m - z_m.min()) * scale * EXAG + BASE_MM
xs = np.linspace(0, EXTENT_X * scale, GX)
ys = np.linspace(EXTENT_Y * scale, 0, GY)    # row 0 = north
X, Y = np.meshgrid(xs, ys)
top = np.dstack([X, Y, z])
bot = np.dstack([X, Y, np.zeros_like(z)])

## Build the mesh

Top surface, flat bottom, and four side walls, each wound so normals point outward.

In [ ]:
tris = []
# top surface (normal up): rows go north->south, so wind accordingly
a = top[:-1, :-1]; b = top[:-1, 1:]; c = top[1:, :-1]; d = top[1:, 1:]
tris.append(np.stack([a, c, b], -2).reshape(-1, 3, 3))
tris.append(np.stack([b, c, d], -2).reshape(-1, 3, 3))
# bottom (normal down)
a = bot[:-1, :-1]; b = bot[:-1, 1:]; c = bot[1:, :-1]; d = bot[1:, 1:]
tris.append(np.stack([a, b, c], -2).reshape(-1, 3, 3))
tris.append(np.stack([b, d, c], -2).reshape(-1, 3, 3))
# side walls
def wall(t_edge, b_edge, flip):
    t0, t1, b0, b1 = t_edge[:-1], t_edge[1:], b_edge[:-1], b_edge[1:]
    if flip:
        return np.concatenate([np.stack([t0, t1, b0], 1), np.stack([t1, b1, b0], 1)])
    return np.concatenate([np.stack([t0, b0, t1], 1), np.stack([t1, b0, b1], 1)])
tris.append(wall(top[0], bot[0], True))        # north edge (x increasing)
tris.append(wall(top[-1], bot[-1], False))       # south edge
tris.append(wall(top[:, 0], bot[:, 0], False))   # west edge (y decreasing)
tris.append(wall(top[:, -1], bot[:, -1], True))# east edge
T = np.concatenate(tris).astype(np.float32)

## Write the binary STL

In [ ]:
n = np.cross(T[:, 1] - T[:, 0], T[:, 2] - T[:, 0])
n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-12
rec = np.zeros(len(T), dtype=[('n', '<f4', 3), ('v', '<f4', (3, 3)), ('a', '<u2')])
rec['n'] = n; rec['v'] = T
with open(f'{NAME}.stl', 'wb') as f:
    f.write(f'Mount St Helens {EXAG:g}x terrain - USGS 3DEP'.encode()[:80].ljust(80, b' '))
    f.write(struct.pack('<I', len(T)))
    rec.tofile(f)

## Sanity check

A positive signed volume means the normals point outward.

In [ ]:
vol = np.einsum('ij,ij->i', T[:, 0], np.cross(T[:, 1], T[:, 2])).sum() / 6
print(f"triangles={len(T)} size={X.max():.1f}x{Y.max():.1f}mm height={z.max():.1f}mm volume={vol/1000:.1f}cm3")

## Hillshade preview

In [ ]:
gy, gx = np.gradient(dem, RES_M)
slope = np.arctan(np.hypot(gx, gy)); aspect = np.arctan2(-gx, gy)
az, alt = np.radians(315), np.radians(45)
hs = np.sin(alt)*np.cos(slope) + np.cos(alt)*np.sin(slope)*np.cos(az - aspect)
Image.fromarray((np.clip(hs, 0, 1)*255).astype(np.uint8)).save(f'preview_hillshade_{NAME}.png')
Image.open(f'preview_hillshade_{NAME}.png')